<a href="https://colab.research.google.com/github/sergioGarcia91/SeismicUP/blob/main/09b_PGA_SGC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inicio

In [ ]:
!git clone https://github.com/sergioGarcia91/SeismicUP.git

In [ ]:
# para incluir la libreria clonada
import sys
sys.path.append("/content/SeismicUP")

import seismicup as sup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, LogLocator

# Cambiar Fuente

In [ ]:
sup.plots.get_TimesNewRoman_font()

# Gráfica

In [ ]:
mw = np.array([1.0, 2.5, 3.6, 4.5, 5.5, 6.4])

distanciaEpi = np.linspace(10, 300, 100) # km
profundidad = np.array([5]) # km

In [ ]:
def PGA_Z_m03(Mw, Repi, H, X=0):
  log10_Repi = np.log10(Repi)
  C = 4.2114
  a = 0.6723
  b = -3.4934
  c = 0.0053
  sigma = 0.2324

  epsilon = X * sigma

  return C + a * Mw + b * log10_Repi + c * H + epsilon


In [ ]:

fig, ax = plt.subplots(ncols=1, nrows=1, figsize=(7,4))

for m in mw:
  pga_z_0 = PGA_Z_m03(Mw = m,
                      Repi= distanciaEpi,
                      H= profundidad[0],
                      X= 1)

  pga_z_0 = 10**pga_z_0

  pga_z_1 = PGA_Z_m03(Mw = m,
                    Repi= distanciaEpi,
                    H= profundidad[0],
                    X= 0)

  pga_z_1 = 10**pga_z_1

  pga_z_2 = PGA_Z_m03(Mw = m,
                    Repi= distanciaEpi,
                    H= profundidad[0],
                    X= -1)

  pga_z_2 = 10**pga_z_2


  line, = ax.plot(distanciaEpi,
          pga_z_0,
          ls='--',
          alpha=0.5)
  color = line.get_color()

  if m >=3.6:
    label_pga = f'Mw = {m}'
  else:
    label_pga = f'Mw = {m} (referencia)'

  ax.plot(distanciaEpi,
          pga_z_1,
          ls='-',
          label=label_pga,
          c=color)

  ax.plot(distanciaEpi,
          pga_z_2,
          ls='--',
          c=color,
          alpha=0.5)

ax.set_xscale('log')
ax.set_yscale('log')

#ax.set_xlim(90, 320)
# Eje X (logarítmico)
ax.xaxis.set_minor_locator(
    LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1)
)

# Eje Y (logarítmico)
ax.yaxis.set_minor_locator(LogLocator(base=10.0,
                                      subs=np.arange(2, 10) * 0.1))
# Activar los ticks menores
ax.minorticks_on()

# Grilla mayor
ax.grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax.grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax.set_xlabel('Distancia epicentral, $R_{epi}$ [km]')
ax.set_ylabel(r"$PGA_Z$ [cm/s$^2$]")

plt.legend()

ax.set_title(f'Curvas de atenuación para $H={profundidad[0]:.0f}$ k ($\pm1\sigma$)')

plt.show()

# Cargar CSV

In [ ]:
df_ = pd.read_csv('/content/SeismicUP/Outputs/20260710_catalogo_PGA_SGC.csv',
                  sep=';',
                  decimal=',')

new_row = pd.DataFrame({
    'Región': ['BRJC'],
    'Latitud': [6.97],
    'Longitud': [-73.74],
    'Profundidad (km)': [108]
})

df_ = pd.concat([df_, new_row], ignore_index=True)

df_.info()

In [ ]:
df_ = pd.read_csv('/content/SeismicUP/Outputs/20260710_catalogo_PGA_SGC.csv',
                  sep=';',
                  decimal=',')

new_row = pd.DataFrame({
    'Región': ['BRJC'],
    'Latitud': [6.97],
    'Longitud': [-73.74],
    'Profundidad (km)': [108]
})

df_ = pd.concat([df_, new_row], ignore_index=True)

df_.info()

# Geodataframe

In [ ]:
# Incluir las coordenadsa para georfrenciar la estacion

In [ ]:
gdf_unido = gpd.GeoDataFrame(
    df_, geometry=gpd.points_from_xy(df_.Longitud, df_.Latitud), crs="EPSG:4326"
)

In [ ]:
gdf_unido.plot()

In [ ]:
gdf_unido = gdf_unido.to_crs(9377)

gdf_unido.plot()

## Obtener X y Y

In [ ]:
gdf_unido.get_coordinates()

In [ ]:
df_['X [km]'] = gdf_unido.get_coordinates()['x'] / 1000
df_['Y [km]'] = gdf_unido.get_coordinates()['y'] / 1000

df_.tail()

In [ ]:
df_.info()

# DF PGA

In [ ]:
df_.columns

In [ ]:
df_PGA = df_[['X [km]', 'Error longitud', 'Y [km]', 'Error latitud',
       'Profundidad (km)', 'Error profundidad (km)', 'Magnitud',
       'Error magnitud', 'Tipo magnitud', 'PGA EW (cm/s^2)', 'PGA NS (cm/s^2)',
       'PGA Z (cm/s^2)', ]].copy()
df_PGA = df_PGA.iloc[:-1]

df_BRJC = df_.iloc[-1].copy()
df_BRJC = df_BRJC[['X [km]', 'Y [km]', 'Profundidad (km)']]

print(df_BRJC, '\n')

df_PGA

# Homogeneizar magnitudes

In [ ]:
# no todas son Mw ... se va usar la formula del SGC para pasar a Mw
# considerar para la de Campo Colorado usar mejor Ml
# que ya lo entrega así el SGC
df_PGA['Tipo magnitud'].unique()

In [ ]:
# Casi todas dicen Ml ... para M no aparece una formula
# se va usar la misma de Ml
# Mw (2.9 <Ml <= 6.1) = 0.958*Ml + 0.1
df_PGA['Mw'] = df_PGA['Magnitud'] * 0.958 + 0.1

# Para dejas Mw igual
df_PGA['Mw'][df_PGA['Tipo magnitud'].str.contains('Mw')] = df_PGA['Magnitud'][df_PGA['Tipo magnitud'].str.contains('Mw')]

df_PGA.head(10)

In [ ]:
df_PGA_aumentado = df_PGA.copy()

# Calculo parametros para PGA

In [ ]:
df_BRJC

In [ ]:
# Distancia eventos a la estacion BRJC
df_PGA_aumentado['Dist Epi [km]'] = np.sqrt(
    (df_PGA_aumentado['X [km]'] - df_BRJC['X [km]'])**2 +
    (df_PGA_aumentado['Y [km]'] - df_BRJC['Y [km]'])**2)

df_PGA_aumentado['Dist Hip [km]'] = np.sqrt(
    (df_PGA_aumentado['X [km]'] - df_BRJC['X [km]'])**2 +
    (df_PGA_aumentado['Y [km]'] - df_BRJC['Y [km]'])**2 +
    (df_PGA_aumentado['Profundidad (km)'] - df_BRJC['Profundidad (km)'])**2)

df_PGA_aumentado.describe().round(2)

# Modelos PGA empiricos Notebook 09

In [ ]:
def PGA_EW_m03(Mw, Repi, H, X=0):
  log10_Repi = np.log10(Repi)
  C = 4.7827
  a = 0.6840
  b = -3.5873
  c = 0.0048
  sigma = 0.2734

  epsilon = X * sigma

  return C + a * Mw + b * log10_Repi + c * H + epsilon

def PGA_NS_m03(Mw, Repi, H, X=0):
  log10_Repi = np.log10(Repi)
  C = 5.0227
  a = 0.6720
  b = -3.6422
  c = 0.0054
  sigma = 0.2594

  epsilon = X * sigma

  return C + a * Mw + b * log10_Repi + c * H + epsilon

def PGA_Z_m03(Mw, Repi, H, X=0):
  log10_Repi = np.log10(Repi)
  C = 4.2114
  a = 0.6723
  b = -3.4934
  c = 0.0053
  sigma = 0.2324

  epsilon = X * sigma

  return C + a * Mw + b * log10_Repi + c * H + epsilon

In [ ]:
df_PGA_aumentado.columns

In [ ]:
df_PGA_aumentado['PGA_EW_cal'] = PGA_EW_m03(Mw = df_PGA_aumentado['Mw'],
                                            Repi= df_PGA_aumentado['Dist Epi [km]'],
                                            H= df_PGA_aumentado['Profundidad (km)'],
                                            X= 0)
df_PGA_aumentado['Residual_EW'] = np.log10(df_PGA_aumentado['PGA EW (cm/s^2)']) - df_PGA_aumentado['PGA_EW_cal']
df_PGA_aumentado['PGA_EW_cal'] = 10**df_PGA_aumentado['PGA_EW_cal']

df_PGA_aumentado['PGA_NS_cal'] = PGA_NS_m03(Mw = df_PGA_aumentado['Mw'],
                                            Repi= df_PGA_aumentado['Dist Epi [km]'],
                                            H= df_PGA_aumentado['Profundidad (km)'],
                                            X= 0)
df_PGA_aumentado['Residual_NS'] = np.log10(df_PGA_aumentado['PGA NS (cm/s^2)']) - df_PGA_aumentado['PGA_NS_cal']
df_PGA_aumentado['PGA_NS_cal'] = 10**df_PGA_aumentado['PGA_NS_cal']

df_PGA_aumentado['PGA_Z_cal'] = PGA_Z_m03(Mw = df_PGA_aumentado['Mw'],
                                          Repi= df_PGA_aumentado['Dist Epi [km]'],
                                          H= df_PGA_aumentado['Profundidad (km)'],
                                          X= 0)
df_PGA_aumentado['Residual_Z'] = np.log10(df_PGA_aumentado['PGA Z (cm/s^2)']) - df_PGA_aumentado['PGA_Z_cal']
df_PGA_aumentado['PGA_Z_cal'] = 10**df_PGA_aumentado['PGA_Z_cal']

## Plots

In [ ]:
# Límites comunes para ambos ejes
xmin = min(df_PGA_aumentado['PGA EW (cm/s^2)'].min(),
           df_PGA_aumentado['PGA_EW_cal'].min())

xmax = max(df_PGA_aumentado['PGA EW (cm/s^2)'].max(),
           df_PGA_aumentado['PGA_EW_cal'].max())

fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(7,7))

ax[0,0].scatter(df_PGA_aumentado['PGA EW (cm/s^2)'],
           df_PGA_aumentado['PGA_EW_cal'],
           s=7,
           alpha=0.7)

# Línea 1:1
ax[0,0].plot([xmin, xmax],
        [xmin, xmax],
        'k-',
        lw=1,
        alpha=0.7,
        label='y = x')

ax[0,0].set_xscale('log')
ax[0,0].set_yscale('log')

ax[0,0].set_xlim(xmin, xmax)
ax[0,0].set_ylim(xmin, xmax)

# Grilla mayor
ax[0,0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0,0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0,0].set_xlabel('Observado')
ax[0,0].set_ylabel('Calculado')
ax[0,0].axis('equal')

# Histograma Residuales
ax[0,1].hist(df_PGA_aumentado['Residual_EW'],
           bins=30)
ax[0,1].grid(linestyle='--', color='lightgray', alpha=0.8)
ax[0,1].set_xlabel(r'Residuales = log$_{10}\frac{observado}{calculado}$')
ax[0,1].axvline(df_PGA_aumentado['Residual_EW'].mean(),
              color='r',
              lw=1,
              label=f'Media = {df_PGA_aumentado["Residual_EW"].mean():.2f}')

ax[0,1].axvline(0,
              color='k',
              linestyle='--',
              lw=1,
              label='Residual = 0')
ax[0,1].legend()


# Residuales frente al Mw
ax[1,0].scatter(
    df_PGA_aumentado['Mw'],
    df_PGA_aumentado['Residual_EW'],
    alpha=0.7
)
ax[1,0].set_xlabel('Mw')
ax[1,0].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,0].grid(linestyle='--', color='lightgray', alpha=0.8)

# Residuales frente al REpi
ax[1,1].scatter(
    df_PGA_aumentado['Dist Epi [km]'],
    df_PGA_aumentado['Residual_EW'],
    alpha=0.7
)
ax[1,1].set_xscale('log')
ax[1,1].set_xlabel('$R_{Epi}$')
ax[1,1].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,1].grid(linestyle='--', color='lightgray', alpha=0.8)

plt.suptitle('PGA EW [$cm/s^2$]')

plt.tight_layout()

plt.show()

In [ ]:
# Límites comunes para ambos ejes
xmin = min(df_PGA_aumentado['PGA NS (cm/s^2)'].min(),
           df_PGA_aumentado['PGA_NS_cal'].min())

xmax = max(df_PGA_aumentado['PGA NS (cm/s^2)'].max(),
           df_PGA_aumentado['PGA_NS_cal'].max())

fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(7,7))

ax[0,0].scatter(df_PGA_aumentado['PGA NS (cm/s^2)'],
           df_PGA_aumentado['PGA_NS_cal'],
           s=7,
           alpha=0.7)

# Línea 1:1
ax[0,0].plot([xmin, xmax],
        [xmin, xmax],
        'k-',
        lw=1,
        alpha=0.7,
        label='y = x')

ax[0,0].set_xscale('log')
ax[0,0].set_yscale('log')

ax[0,0].set_xlim(xmin, xmax)
ax[0,0].set_ylim(xmin, xmax)

# Grilla mayor
ax[0,0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0,0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0,0].set_xlabel('Observado')
ax[0,0].set_ylabel('Calculado')
ax[0,0].axis('equal')

# Histograma Residuales
ax[0,1].hist(df_PGA_aumentado['Residual_NS'],
           bins=30)
ax[0,1].grid(linestyle='--', color='lightgray', alpha=0.8)
ax[0,1].set_xlabel(r'Residuales = log$_{10}\frac{observado}{calculado}$')
ax[0,1].axvline(df_PGA_aumentado['Residual_NS'].mean(),
              color='r',
              lw=1,
              label=f'Media = {df_PGA_aumentado["Residual_NS"].mean():.2f}')

ax[0,1].axvline(0,
              color='k',
              linestyle='--',
              lw=1,
              label='Residual = 0')
ax[0,1].legend()


# Residuales frente al Mw
ax[1,0].scatter(
    df_PGA_aumentado['Mw'],
    df_PGA_aumentado['Residual_NS'],
    alpha=0.7
)
ax[1,0].set_xlabel('Mw')
ax[1,0].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,0].grid(linestyle='--', color='lightgray', alpha=0.8)

# Residuales frente al REpi
ax[1,1].scatter(
    df_PGA_aumentado['Dist Epi [km]'],
    df_PGA_aumentado['Residual_NS'],
    alpha=0.7
)
ax[1,1].set_xscale('log')
ax[1,1].set_xlabel('$R_{Epi}$')
ax[1,1].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,1].grid(linestyle='--', color='lightgray', alpha=0.8)

plt.suptitle('PGA NS [$cm/s^2$]')

plt.tight_layout()

plt.show()

In [ ]:
# Límites comunes para ambos ejes
xmin = min(df_PGA_aumentado['PGA Z (cm/s^2)'].min(),
           df_PGA_aumentado['PGA_Z_cal'].min())

xmax = max(df_PGA_aumentado['PGA Z (cm/s^2)'].max(),
           df_PGA_aumentado['PGA_Z_cal'].max())

fig, ax = plt.subplots(ncols=2, nrows=2, figsize=(7,7))

ax[0,0].scatter(df_PGA_aumentado['PGA Z (cm/s^2)'],
           df_PGA_aumentado['PGA_Z_cal'],
           s=7,
           alpha=0.7)

# Línea 1:1
ax[0,0].plot([xmin, xmax],
        [xmin, xmax],
        'k-',
        lw=1,
        alpha=0.7,
        label='y = x')

ax[0,0].set_xscale('log')
ax[0,0].set_yscale('log')

ax[0,0].set_xlim(xmin, xmax)
ax[0,0].set_ylim(xmin, xmax)

# Grilla mayor
ax[0,0].grid(which='major', linestyle='--', color='gray', alpha=0.8)

# Grilla menor
ax[0,0].grid(which='minor', linestyle='--', color='lightgray', alpha=0.8)

ax[0,0].set_xlabel('Observado')
ax[0,0].set_ylabel('Calculado')
ax[0,0].axis('equal')

# Histograma Residuales
ax[0,1].hist(df_PGA_aumentado['Residual_Z'],
           bins=30)
ax[0,1].grid(linestyle='--', color='lightgray', alpha=0.8)
ax[0,1].set_xlabel(r'Residuales = log$_{10}\frac{observado}{calculado}$')
ax[0,1].axvline(df_PGA_aumentado['Residual_Z'].mean(),
              color='r',
              lw=1,
              label=f'Media = {df_PGA_aumentado["Residual_Z"].mean():.2f}')

ax[0,1].axvline(0,
              color='k',
              linestyle='--',
              lw=1,
              label='Residual = 0')
ax[0,1].legend()


# Residuales frente al Mw
ax[1,0].scatter(
    df_PGA_aumentado['Mw'],
    df_PGA_aumentado['Residual_Z'],
    alpha=0.7
)
ax[1,0].set_xlabel('Mw')
ax[1,0].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,0].grid(linestyle='--', color='lightgray', alpha=0.8)

# Residuales frente al REpi
ax[1,1].scatter(
    df_PGA_aumentado['Dist Epi [km]'],
    df_PGA_aumentado['Residual_Z'],
    alpha=0.7
)
ax[1,1].set_xscale('log')
ax[1,1].set_xlabel('$R_{Epi}$')
ax[1,1].set_ylabel(r'Residuales = $log_{10}​(PGA_{obs}​)−log_{10}​(PGA_{cal}​)$')
ax[1,1].grid(linestyle='--', color='lightgray', alpha=0.8)

plt.suptitle('PGA Z [$cm/s^2$]')

plt.tight_layout()

plt.show()

# Fin